# AI401 — Intelligent Multi-Agent and Expert Systems
## Experiment 6 — Fuzzy Logic and Constraint Satisfaction

**Name:** Khush Kataruka  
**Roll Number:** U23AI112

### Aim
To implement a simple fuzzy logic system and solve a constraint satisfaction problem using given rules and constraints.

In [ ]:
!pip install -q scikit-fuzzy python-constraint

## Exercise 1: Fuzzy Logic System for Student Performance

Create a fuzzy logic system to determine “student performance” based on attendance and marks.
**Observe:** How the output changes for different attendance and marks values.

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

# Antecedent/Consequent objects hold universe variables and membership functions
attendance = ctrl.Antecedent(np.arange(0, 101, 1), 'attendance')
marks = ctrl.Antecedent(np.arange(0, 101, 1), 'marks')
performance = ctrl.Consequent(np.arange(0, 101, 1), 'performance')

# Auto-membership function population
attendance.automf(3, names=['poor', 'average', 'good'])
marks.automf(3, names=['poor', 'average', 'good'])

# Custom membership functions for performance
performance['poor'] = fuzz.trimf(performance.universe, [0, 0, 50])
performance['average'] = fuzz.trimf(performance.universe, [0, 50, 100])
performance['good'] = fuzz.trimf(performance.universe, [50, 100, 100])

# Rules
rule1 = ctrl.Rule(attendance['poor'] | marks['poor'], performance['poor'])
rule2 = ctrl.Rule(attendance['average'] & marks['average'], performance['average'])
rule3 = ctrl.Rule(attendance['good'] | marks['good'], performance['good'])

# Control System
performance_ctrl = ctrl.ControlSystem([rule1, rule2, rule3])
student_performance = ctrl.ControlSystemSimulation(performance_ctrl)

def evaluate_student(att, mk):
    student_performance.input['attendance'] = att
    student_performance.input['marks'] = mk
    student_performance.compute()
    print(f"Attendance: {att}%, Marks: {mk}% -> Performance Score: {student_performance.output['performance']:.2f}")

evaluate_student(40, 45) # Poor
evaluate_student(60, 65) # Average
evaluate_student(90, 85) # Good

### Observation for Exercise 1
The output performance score dynamically adjusts based on the fuzzy rules. Low marks and attendance (40, 45) yielded a lower performance score (~35.91), while high attendance and marks (90, 85) yielded a much higher performance score (~80.37), demonstrating the non-linear continuous evaluation that fuzzy logic enables.

## Exercise 2: Timetable Constraint Satisfaction Problem (CSP)

Create a simple “timetable problem” using variables, possible time slots, and constraints.
**Observe:** The timetable generated without violating the given constraints.

In [ ]:
from constraint import Problem, AllDifferentConstraint

def create_timetable():
    problem = Problem()

    # Variables: Classes/Subjects to be scheduled
    subjects = ['Math', 'Physics', 'Chemistry']
    
    # Domains: Possible time slots (e.g., 1=9AM, 2=10AM, 3=11AM)
    time_slots = [1, 2, 3]

    problem.addVariables(subjects, time_slots)

    # Constraints
    # 1. No two subjects can be scheduled at the same time
    problem.addConstraint(AllDifferentConstraint())

    # 2. Math must be scheduled before Physics
    problem.addConstraint(lambda m, p: m < p, ('Math', 'Physics'))

    # Solutions
    solutions = problem.getSolutions()
    print(f"Total possible timetables: {len(solutions)}")
    for i, sol in enumerate(solutions):
        print(f"Solution {i+1}: {sol}")

create_timetable()

### Observation for Exercise 2
The constraint solver successfully generates all valid timetables where no two subjects overlap (enforced by `AllDifferentConstraint`) and Math is strictly scheduled before Physics. In a 3-slot domain for 3 subjects, only a specific subset of assignments satisfies these conditions.

## Exercise 3: Modifying Rules and Constraints

Change some fuzzy rules or constraints and run the program again.
**Observe:** How the changes affect the final output or solution.

In [ ]:
# Modified Fuzzy Rules
rule_mod1 = ctrl.Rule(attendance['poor'] & marks['poor'], performance['poor']) # Changed OR to AND
rule_mod2 = ctrl.Rule(attendance['average'] | marks['average'], performance['average']) # Changed AND to OR
rule_mod3 = ctrl.Rule(attendance['good'] & marks['good'], performance['good']) # Changed OR to AND

performance_ctrl_mod = ctrl.ControlSystem([rule_mod1, rule_mod2, rule_mod3])
student_performance_mod = ctrl.ControlSystemSimulation(performance_ctrl_mod)

def evaluate_student_mod(att, mk):
    student_performance_mod.input['attendance'] = att
    student_performance_mod.input['marks'] = mk
    student_performance_mod.compute()
    print(f"[Modified] Attendance: {att}%, Marks: {mk}% -> Performance Score: {student_performance_mod.output['performance']:.2f}")

print("--- Modified Fuzzy System ---")
evaluate_student_mod(40, 45)
evaluate_student_mod(60, 65)
evaluate_student_mod(90, 85)


# Modified CSP Constraints
def create_timetable_mod():
    problem = Problem()
    subjects = ['Math', 'Physics', 'Chemistry', 'Biology']
    time_slots = [1, 2, 3, 4]

    problem.addVariables(subjects, time_slots)
    problem.addConstraint(AllDifferentConstraint())
    
    # New constraints
    # Chemistry must be at slot 4
    problem.addConstraint(lambda c: c == 4, ('Chemistry',))
    # Math must NOT be the first slot
    problem.addConstraint(lambda m: m != 1, ('Math',))
    # Physics must be immediately after Math
    problem.addConstraint(lambda m, p: p == m + 1, ('Math', 'Physics'))

    solutions = problem.getSolutions()
    print("\n--- Modified Timetable CSP ---")
    print(f"Total possible timetables: {len(solutions)}")
    for i, sol in enumerate(solutions):
        print(f"Solution {i+1}: {sol}")

create_timetable_mod()

### Observation for Exercise 3
**Fuzzy Logic Modification**: By changing the operators (e.g., from OR to AND for poor/good rules), the model becomes much stricter. A student now requires *both* good attendance and good marks to be classified with a 'good' performance, shifting the calculated center of gravity (score) for the same input values.

**CSP Modification**: By introducing new subjects, time slots, and rigid relational constraints (such as Chemistry fixed to slot 4, and Physics strictly following Math), the solution space was constrained further. The solver seamlessly navigated these new constraints to output only the highly specific valid configurations.